# Verificación — Spark lee tu datalake

**Curso:** ST1630-2026-2 · **Semana:** S4-S5
**Estudiante:** Juan José Díaz Rodríguez (`jjdiazr`)
**Fecha de ejecución:** 2026-08-13
**Clúster:** `j-28Y8PCM6OPLOA` · emr-6.15.0 · Spark 3.4.1 · 2 × m5.xlarge
**Bucket:** `s3://st1630-jjdiazr-2026`

## Objetivo

Cerrar el Lab 1a confirmando que tu clúster EMR puede leer el datalake
que construiste (Partes 1-4): conectar Spark a tu bucket S3, leer el
archivo Parquet de Bronze, y repetir el benchmark Parquet vs. CSV visto
en la clase de S4.

**Qué debe verse al final para confirmar que el lab está completo:**
- La Celda 2 muestra el schema y 5 filas del Parquet leído desde S3
  (si esto funciona, tu bucket, tu rol IAM y tu clúster están bien
  configurados de punta a punta).
- La Celda 3 imprime el tiempo de una misma consulta en Parquet y en
  CSV, y el ratio entre ambos.
- Completaste el análisis de la Celda 4 y capturaste el DAG de Spark UI
  como indica la Celda 5.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# getOrCreate() funciona tanto en EMR (donde ya existe una sesión activa
# administrada por el clúster) como en un entorno local con pyspark
# instalado, sin necesitar ramas de código distintas.
spark = SparkSession.builder.appName("ST1630-Lab1a-Verificacion").getOrCreate()

# EDITAR: reemplaza por el bucket que creaste en setup_s3.sh
# (convención: st1630-{tu-usuario}-{año})
BUCKET = "st1630-jjdiazr-2026"

# En EMR, S3 se referencia directamente con el esquema s3://.
# En local (con las credenciales de AWS Academy exportadas), la misma
# ruta también funciona porque Spark usa el conector S3A por debajo.
ruta_parquet = f"s3://{BUCKET}/bronze/ventas/prueba_parquet.parquet"
ruta_csv = f"s3://{BUCKET}/bronze/ventas/prueba_csv.csv"

df_parquet = spark.read.parquet(ruta_parquet)

df_parquet.printSchema()
df_parquet.show(5, truncate=False)

# Si ves el schema y las filas de arriba, tu datalake funciona
# correctamente de punta a punta: bucket, permisos IAM y clúster EMR.
print("Filas leídas:", df_parquet.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 16:06:15 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


root
 |-- order_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- region: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: long (nullable = true)
 |-- precio_unit: double (nullable = true)
 |-- total: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- devuelto: boolean (nullable = true)



+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
|order_id  |fecha     |region      |producto |categoria  |cantidad|precio_unit|total    |canal |devuelto|
+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
|ORD-000001|2026-04-22|Cali        |Mouse    |Electrónica|1       |789300.0   |789300.0 |online|false   |
|ORD-000002|2026-03-22|Barranquilla|Gorra    |Ropa       |2       |57000.0    |114000.0 |tienda|false   |
|ORD-000003|2026-01-06|Bogotá      |Zapatos  |Ropa       |4       |163800.0   |655200.0 |online|false   |
|ORD-000004|2025-11-26|Barranquilla|Panela   |Alimentos  |3       |54900.0    |164700.0 |online|false   |
|ORD-000005|2026-01-07|Cali        |Audífonos|Electrónica|4       |251100.0   |1004400.0|tienda|true    |
+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
only showing top 5 rows

Filas leídas: 10000


In [2]:
import time

df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(ruta_csv)

# Misma consulta sobre ambos formatos: filtrar por región y categoría,
# y agregar el total vendido -- el tipo de consulta selectiva que se
# beneficia de predicado pushdown y column pruning en formatos
# columnares (visto en la clase de S4, slide de Parquet vs. CSV).

def benchmark(df, nombre):
    inicio = time.time()
    resultado = (
        df.filter((F.col("region") == "Bogotá") & (F.col("categoria") == "Electrónica"))
          .groupBy("producto")
          .agg(F.sum("total").alias("total_vendido"))
          .orderBy(F.col("total_vendido").desc())
          .collect()  # acción -- fuerza la ejecución real, no solo el plan
    )
    duracion = time.time() - inicio
    print(f"{nombre}: {duracion:.3f} s ({len(resultado)} filas de resultado)")
    return duracion

tiempo_parquet = benchmark(df_parquet, "Parquet")
tiempo_csv = benchmark(df_csv, "CSV")

ratio = tiempo_csv / tiempo_parquet if tiempo_parquet > 0 else float("inf")
print(f"\nRatio CSV / Parquet: {ratio:.2f}x")

Parquet: 1.666 s (6 filas de resultado)
CSV: 0.851 s (6 filas de resultado)

Ratio CSV / Parquet: 0.51x


## Análisis

### a) Tamaño en disco

¿Cuánto pesa `prueba_parquet.parquet` frente a `prueba_csv.csv`?

→ **Parquet 185.4 KiB vs CSV 798.3 KiB** — el CSV ocupa **4.3 veces más** con
exactamente los mismos datos (10.000 filas, 10 columnas). Confirmado tanto en la
salida de `generar_datos.py` como con `aws s3 ls --human-readable` sobre
`bronze/ventas/`.

La razón está en cómo guarda cada formato. Parquet almacena por columnas, así que
quedan juntos valores del mismo tipo y muy repetidos: la columna `region` son 5
ciudades repetidas 10.000 veces, y eso se comprime muy bien (snappy). El CSV
guarda por filas, mezclando en cada línea texto, enteros, decimales y booleanos,
y además escribe todos los números como texto. Parquet también guarda el schema
dentro del archivo — por eso la Celda 2 leyó los tipos correctos (`cantidad` como
`long`, `devuelto` como `boolean`) sin que yo se los indicara, mientras que el CSV
necesitó `inferSchema=true`, que obliga a Spark a leer el archivo una vez extra
solo para adivinarlos.

### b) Tiempo de la consulta

¿Cuánto tardó la consulta de la Celda 3 en cada formato?

→ **Parquet 1.666 s · CSV 0.851 s.** Es decir, el CSV fue casi el doble de rápido
que el Parquet — lo contrario de lo esperado.

### c) Ratio de mejora

¿Cuál fue el ratio (CSV / Parquet)? ¿Coincide con el orden de magnitud visto en
clase (~9x)?

→ **0.51x**, muy lejos del ~9x de clase y además en la dirección contraria. No
repetí la medición hasta que diera "bonito": el número es real y lo que importa es
explicar por qué salió así.

La causa principal es **el orden de ejecución**. El benchmark de Parquet corrió
primero, y ese primer job pagó todo el arranque de Spark en el clúster: solicitar
contenedores a YARN, levantar los ejecutores y subir las librerías de Spark a
HDFS — el aviso `WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is
set` que aparece en la salida de la Celda 2 es exactamente eso. Cuando le tocó el
turno al CSV, el clúster ya estaba caliente y solo midió la consulta.

El segundo factor es **el tamaño del dataset**. Con 10.000 filas (menos de 1 MB),
el trabajo real de leer y filtrar se cuenta en milisegundos, mientras que los
costos fijos — planificar el job, agendar tareas, abrir conexiones a S3, hacer el
shuffle del `Exchange` — se cuentan en cientos de milisegundos. A esta escala el
costo fijo domina por completo y tapa cualquier ventaja del formato. Las ventajas
de Parquet (column pruning: mi consulta usa 4 de 10 columnas; predicate pushdown
al filtrar por `region` y `categoria`) solo se notan cuando hay suficientes datos
que evitar leer. Aquí no los hay.

Un tercer detalle juega también en contra de la comparación: el `inferSchema=true`
del CSV ya había leído el archivo completo al construir el DataFrame, **fuera del
cronómetro**, así que parte de su costo real quedó sin medir.

Cómo lo mediría bien: una ronda de calentamiento sobre ambos formatos antes de
cronometrar, alternando el orden entre rondas, repitiendo varias veces y
comparando medianas — y sobre todo, con un dataset de al menos algunos GB, que es
la escala donde la diferencia entre columnar y de filas es la que se discutió en
clase.

### d) Conexión con el Teorema CAP

¿Por qué S3 con replicación entre múltiples zonas es una decisión **CP**? ¿Qué
sacrifica a cambio?

→ Es **CP** porque S3 nunca entrega una escritura a medias: o el objeto quedó
replicado y confirmado, o no quedó. Cuando escribo un objeto, S3 no responde
"listo" hasta que el dato está durablemente replicado en varias zonas de
disponibilidad, y cualquier lectura posterior ve esa versión y no una anterior
(consistencia de lectura tras escritura, garantizada desde 2020).

Ante una partición de red entre zonas — la **P**, que en un sistema distribuido
real no es opcional: las particiones ocurren y hay que tolerarlas — S3 tiene que
elegir entre confirmar la escritura con el dato en una sola zona, arriesgando que
alguien lea una versión desactualizada desde otra, o esperar hasta poder replicar.
Elige esperar.

Lo que sacrifica, entonces, es la **A de Availability**: durante la partición la
escritura no se completa de inmediato, se demora o falla. S3 prefiere hacerme
esperar antes que confirmarme algo que otro podría leer viejo. Un sistema AP haría
lo contrario — aceptaría la escritura siempre y reconciliaría después —, lo cual
está bien para un carrito de compras pero sería inaceptable para la capa Bronze de
un datalake, donde todo lo demás se reconstruye a partir de ese dato: si `bronze/`
puede devolver una versión desactualizada, entonces `silver/` y `gold/` heredan el
error sin que nadie lo note.

## Captura del DAG en Spark UI

1. En EMR Studio (o en la consola de tu clúster), abre **Spark UI /
   History Server**.
2. Busca el job correspondiente a la Celda 3 (el `groupBy` + `agg` +
   `orderBy` sobre el Parquet).
3. Abre la pestaña **SQL / DataFrame** y captura una imagen del plan
   (o del DAG visual) que incluya al menos un nodo **Exchange**.
4. Guarda la captura como `dag_spark_ui.png` dentro de tu carpeta de
   entrega y referencíala en tu PR.

**Verifica:** la captura debe mostrar el nombre de tu aplicación
(`ST1630-Lab1a-Verificacion`, definido en la Celda 2) para que quede
claro que es tu propia ejecución.

## Bitácora de delegación

Agente usado: **Claude Code (Opus 5)**, sesión guiada del 2026-08-12 al 2026-08-13.
Ver también la bitácora completa del lab en `architecture.md`, sección 7.

| Tarea | ¿Delegado a agente? | Herramienta | Justificación |
|---|---|---|---|
| Boilerplate de SparkSession / lectura de S3 | **No** | — | Venía resuelto en el notebook base del repo; solo edité la variable `BUCKET`. |
| Método de ejecución del notebook en el clúster | **Sí** | Claude Code | El agente definió cómo correrlo contra YARN (`nbconvert --execute` con `SPARK_HOME` y el `PYTHONPATH` de py4j) tras descartar EMR Studio. Yo ejecuté los comandos en el nodo master. |
| Diseño de la consulta del benchmark (Celda 3) | **No** | — | Venía dada en el notebook base. |
| Interpretación de los resultados (Celda 4) | **Parcial** | Claude Code | La causa del 0.51x (el arranque del clúster dentro del cronómetro) la identificamos en conversación; el agente aportó el dato del costo fijo por archivo y redactó el texto final. La decisión de reportar el número real en vez de repetir la medición fue mía. |
| Respuesta sobre CAP (pregunta d) | **No** | — | Razonamiento propio: S3 o entrega todo o no entrega nada, y espera en vez de responder con datos viejos, luego lo que sacrifica es disponibilidad. El agente lo redactó a partir de esa idea. |
| Captura del DAG en Spark UI | **No** | — | Túnel SSH, navegación por el History Server y captura hechas por mí. |
| Troubleshooting de errores de entorno y conexión | **Sí** | Claude Code | Rutas `/tmp` de Git Bash no resueltas por `aws.exe`, `--use-default-roles` duplicando el instance profile, apertura del puerto 22 en el security group del master. Explícitamente delegable según la rúbrica del lab. |

> La interpretación de los resultados y la conexión con CAP reflejan mi propio
> razonamiento; la redacción final es del agente, según permite `politica-ia.md`.